<a href="https://colab.research.google.com/github/jmora420-hue/Ciencia_de_datos/blob/main/Ejercicio_medidas_de_tendencia_central.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Objetivo
Comprender cómo las palabras funcionales pueden dominar un conteo y ocultar los temas centrales de un discurso.

#AGENDA
1. Preparacion
2. Medidas estadística central
3. Identificación de palabras con mayor frecuencia de aparición
4. Eliminación de palabras (stopwords)
5. Comparación de resultados

###1. Preparación

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")#Color de paleta
ARCHIVO="/content/conteo_palabras_video_youtube.xlsx"
completo=pd.read_excel(ARCHIVO, sheet_name="Conteo completo",header=1)
depurado=pd.read_excel(ARCHIVO, sheet_name="Conteo depurado",header=1)
excluidas=pd.read_excel(ARCHIVO, sheet_name="Palabras excluidas",header=1)

for df in (completo,depurado,excluidas):
  df.columns=[str(c).strip() for c in df.columns]

completo=completo.dropna(subset=["Palabra","Frecuencia"]).copy()
depurado=depurado.dropna(subset=["Palabra","Frecuencia"]).copy()
completo["Frecuencia"]= completo["Frecuencia"].astype(int)
depurado["Frecuencia"]= depurado["Frecuencia"].astype(int)

print(f"Palabras totales: {completo["Frecuencia"].sum():,}")
print(f"Palabras únicas: {len(completo):,}")
display(completo.head(10))



Palabras totales: 4,437
Palabras únicas: 1,154


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


,Palabra,Frecuencia
0,que,309
1,de,215
2,a,108
3,la,106
4,no,97
5,y,97
6,el,86
7,lo,84
8,es,71
9,en,64


###2. Medidas de tendencia central
|Medida| Definición | Utilidad |
|---|---|---|
|Media| Suma de frecuencias dividida entre palabras únicas|Frecuencia promedio, sensible a palabras extremadamente repetidas|
|Mediana| Valor central de las frecuencias ordenadas|Describe mejor una distribución no simétrica|
|Moda|Frecuencia que ocurre en más palabras|Mostrar muchas palabras que aparecen una vez|
|Cuartiles|Dividen los datos ordenados en 4 partes|Localizar el 25%, 50% y 75% de las frecuencias|
|Varianza|Promedio de desviaciones cuadráticas con respecto a la media|Cualifica la dispersión
|Desviación estándar|raíz de la varianza|Dispersión en las mismas unidades de frecuencia|
|IQR - Rango Intercuartilíco|Q3-Q1|Mide una dispersión más robusta del 50% central|
|Asimetría|Mide la falta de simetría|Un valor alto revela una cola de palabras muy frecuentes|

In [ ]:
def gini(valores):
  x=np.sort(np.asarray(valores,dtype=float))
  if len(x)==0 or x.sum()==0:
    return np.nan
  n=len(x)
  return (2*np.sum(np.arange(1,n+1)*x)/(n*x.sum()))-(n+1)/n

def entropia_normalizada(valores):
  p=np.asarray(valores,dtype=float)
  p=p/p.sum()
  h=-(p*np.log2(p).sum())
  return h/np.log2(len(p)) if len(p)>1 else 0.0

**TAREA**
La Impureza de Gini mide la probabilidad de clasificar incorrectamente un elemento elegido al azar si se le asignara una etiqueta aleatoria, es un valor entre 0 a 0.5, y se realiza mediante cálculos de probabilidad estándar, como sumas y multiplicaciones, lo que requiere poco poder de procesamiento.
La entropía mide el nivel de incertidumbre de los datos mediante logaritmos, la Entropía normalizada es un ajuste a esa métrica al dividir la ganancia de la información por la entropía intrínseca de la división, evitando crear muchas categorías distintas, tiene un valor entre 0 a 1

In [ ]:
def resumen_estadistico(df,nombre):
  f =df["Frecuencia"]
  modos =f.mode().tolist()
  return pd.Series({
      "Corpus":nombre,
      "Palabras totales (Tokens)":int(f.sum()),
      "Palabras únicas": int(f.size),
      "Media": f.mean(),
      "Mediana": f.median(),
      "Moda": ", ".join(map(str,modos)),
      "Mínimo":f.min(),
      "Q1(25%)":f.quantile(.25),
      "Q2(50%)":f.quantile(.5),
      "Q3(75%)":f.quantile(.75),
  })
